# 🎁 Notebook 4 — BONUS : Shortcut Learning via la Feature `Month`
## *When Machine Learning Fails — Second Failure Mode: Spurious Seasonal Correlation*
---

**Projet principal** : Class Imbalance Failure (Notebooks 1–3)  
**Ce notebook** : Mode d'échec bonus — Shortcut Learning / Corrélation Spurieuse sur `Month`

### 🎯 Objectif scientifique

Ce notebook explore brièvement un **second mode d'échec** indépendant du déséquilibre de classes :  
> *Les classificateurs non-linéaires peuvent exploiter la variable `Month` comme un **raccourci saisonnier** — une corrélation statistique avec `Revenue` qui n'est pas nécessairement causale ni stable sous un changement de distribution temporelle.*

La structure suit les 4 étapes requises par le PDF du projet :
1. Question de recherche
2. Symptôme observé
3. Hypothèse causale + expérience contrôlée
4. Correction et évaluation

### ❓ Question de recherche

> *Les classificateurs non-linéaires entraînés sur le dataset Online Shoppers exploitent-ils la feature `Month` comme un raccourci, au point d'atteindre une bonne performance en distribution mais de devenir moins robustes sous une évaluation basée sur des mois non vus, et est-ce que supprimer `Month` améliore cette robustesse ?*

### 🧪 Hypothèse causale

> *La feature `Month` peut encoder une corrélation saisonnière avec le comportement d'achat plutôt qu'un signal comportemental stable. Les modèles non-linéaires (Random Forest, LightGBM, MLP) peuvent exploiter cette corrélation comme raccourci. Si l'hypothèse est correcte : (1) les features dérivées de `Month` apparaîtront comme des prédicteurs importants, (2) les performances varieront fortement selon les mois, et (3) les modèles AVEC `Month` seront moins robustes sur des mois non vus que ceux SANS `Month`.*

### ✅ Critère de falsifiabilité

> Si `Month` n'est pas important, si la performance ne change pas selon les mois, et si retirer `Month` n'améliore pas la robustesse OOD, l'hypothèse de shortcut learning n'est pas soutenue.

## 1. Setup et Imports

In [ ]:
!pip install -q lightgbm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import pickle
import io

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

import lightgbm as lgb

np.random.seed(42)
RANDOM_STATE = 42

print("✅ Librairies chargées.")
print(f"   numpy    : {np.__version__}")
print(f"   pandas   : {pd.__version__}")
print(f"   lightgbm : {lgb.__version__}")


## 2. Chargement du Dataset & EDA

> ⚠️ L'EDA est réalisée sur le dataset complet à des fins purement descriptives. Aucun prétraitement appris ici.

In [ ]:
# ── Chargement ──────────────────────────────────────────────
df = pd.read_csv('online_shoppers_intention.csv')
print(f"📊 Dimensions : {df.shape}")

# Suppression des doublons
dup = df.duplicated().sum()
if dup > 0:
    df.drop_duplicates(inplace=True)
    print(f"   → {dup} doublons supprimés. Nouvelle taille : {df.shape}")

print("\n=== Aperçu des 10 premières lignes ===")
display(df.head(10))


In [ ]:
# ── Informations générales ──────────────────────────────────
print("=== df.info() ===")
df.info()

print("\n=== Statistiques descriptives ===")
display(df.describe())

print("\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante ✅")

# ── Distribution cible ───────────────────────────────────────
print("\n=== Distribution de Revenue ===")
rev = df['Revenue'].value_counts()
print(rev)
print(f"Taux d'achat : {rev[True]/len(df)*100:.1f}%")


In [ ]:
# ── EDA Plots ───────────────────────────────────────────────
MONTH_ORDER = ['Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
months_present = [m for m in MONTH_ORDER if m in df['Month'].unique()]

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle("EDA — Focus sur la feature Month et le comportement d'achat",
             fontsize=14, fontweight='bold')

# 1. Distribution de Revenue
ax = axes[0, 0]
df['Revenue'].value_counts().plot(kind='bar', ax=ax, color=['#3498db','#e74c3c'], edgecolor='black')
ax.set_title("Distribution de Revenue", fontweight='bold')
ax.set_xticklabels(['False (Non-achat)','True (Achat)'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)',
                (p.get_x()+p.get_width()/2, p.get_height()+30), ha='center', fontsize=9)

# 2. Nombre de sessions par mois
ax = axes[0, 1]
month_counts = df.groupby('Month').size().reindex(months_present)
month_counts.plot(kind='bar', ax=ax, color='#2ecc71', edgecolor='black')
ax.set_title("Nombre de Sessions par Mois", fontweight='bold')
ax.set_xticklabels(months_present, rotation=45, ha='right')
ax.set_ylabel("Nombre de sessions")

# 3. ⭐ Taux d'achat par mois
ax = axes[0, 2]
month_rate = df.groupby('Month')['Revenue'].mean().reindex(months_present) * 100
bars = month_rate.plot(kind='bar', ax=ax, edgecolor='black')
# Colorier les barres selon le taux
colors_map = plt.cm.RdYlGn(month_rate.values / month_rate.max())
for bar, c in zip(ax.patches, colors_map):
    bar.set_facecolor(c)
ax.set_title("⭐ Taux d'Achat par Mois (%)", fontweight='bold', color='darkred')
ax.set_xticklabels(months_present, rotation=45, ha='right')
ax.set_ylabel("Taux d'achat (%)")
ax.axhline(df['Revenue'].mean()*100, color='navy', linestyle='--',
           label=f'Moyenne globale ({df["Revenue"].mean()*100:.1f}%)')
ax.legend(fontsize=8)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x()+p.get_width()/2, p.get_height()+0.3),
                ha='center', fontsize=7)

# 4. Stacked bar : Revenue par mois
ax = axes[1, 0]
month_rev_stacked = df.groupby(['Month','Revenue']).size().unstack().reindex(months_present)
month_rev_stacked.plot(kind='bar', stacked=True, ax=ax,
                        color=['#3498db','#e74c3c'], edgecolor='black')
ax.set_title("Distribution Revenue par Mois (stacked)", fontweight='bold')
ax.set_xticklabels(months_present, rotation=45, ha='right')
ax.legend(['Non-achat','Achat'], loc='upper right')

# 5. PageValues par Revenue
ax = axes[1, 1]
df[df['Revenue']==False]['PageValues'].clip(upper=150).hist(
    ax=ax, bins=40, alpha=0.6, color='#3498db', label='Non-achat', density=True)
df[df['Revenue']==True]['PageValues'].clip(upper=150).hist(
    ax=ax, bins=40, alpha=0.6, color='#e74c3c', label='Achat', density=True)
ax.set_title("Distribution PageValues par Revenue", fontweight='bold')
ax.set_xlabel("PageValues (clip@150)")
ax.legend()

# 6. Heatmap corrélation
ax = axes[1, 2]
num_cols = ['Administrative','Administrative_Duration','Informational',
            'Informational_Duration','ProductRelated','ProductRelated_Duration',
            'BounceRates','ExitRates','PageValues','SpecialDay']
sns.heatmap(df[num_cols].corr(), ax=ax, cmap='coolwarm', annot=False,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title("Heatmap de Corrélation (Numériques)", fontweight='bold')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('bonus_month_purchase_rate.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée → bonus_month_purchase_rate.png")


### 📌 Interprétation EDA — Taux d'achat par mois

La visualisation du taux d'achat par mois révèle une **variabilité saisonnière significative** :

- **Mois à fort taux d'achat** : Novembre et Décembre montrent généralement des taux d'achat plus élevés, probablement liés aux fêtes de fin d'année (Black Friday, Noël).
- **Mois à faible taux d'achat** : Certains mois d'été présentent des taux plus faibles.

### 🚨 Pourquoi `Month` peut devenir un raccourci ?

Un classificateur non-linéaire peut apprendre que « si c'est Novembre → forte probabilité d'achat ». Cette règle est **statistiquement vraie dans le dataset d'entraînement**, mais elle est:
1. **Non causale** : Le mois lui-même ne cause pas l'achat — ce sont les promotions, le comportement des visiteurs et le contexte commercial qui le font.
2. **Non stable** : Une corrélation saisonnière peut disparaître si les données proviennent d'une autre année, d'un autre site ou si la stratégie commerciale change.
3. **Potentiellement trompeuse** : Un modèle qui apprend ce raccourci fonctionnera bien sur le dataset actuel mais échouera en déploiement sous un shift temporel.

> Ce phénomène est analogue au classifieur husky-vs-loup qui apprend à détecter la neige en arrière-plan — le modèle exploite une corrélation parasitaire plutôt qu'un signal causal robuste.

## 3. Prétraitement Anti-Fuite

> **Règle absolue** : Aucun préprocesseur n'est fitté avant le split. StandardScaler et OneHotEncoder sont fittés uniquement sur X_train.

In [ ]:
# ── Définition des features ─────────────────────────────────
NUM_FEATURES = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]
LOG_FEATURES = [
    'Administrative_Duration', 'Informational_Duration',
    'ProductRelated_Duration', 'ProductRelated', 'PageValues'
]
CAT_WITH_MONTH    = ['Month', 'VisitorType', 'Weekend']
CAT_WITHOUT_MONTH = ['VisitorType', 'Weekend']

# ── Conversion de la cible ───────────────────────────────────
df['Revenue'] = df['Revenue'].astype(int)

# ── Transformation log1p (déterministe, non paramétrique) ───
for feat in LOG_FEATURES:
    df[feat] = np.log1p(df[feat])

print("✅ Transformation log1p appliquée aux features asymétriques.")
print(f"   Features transformées : {LOG_FEATURES}")

# ── Construction des deux préprocesseurs ────────────────────
def make_preprocessor(cat_features):
    """
    Construit un ColumnTransformer complet :
    - SimpleImputer + StandardScaler pour les numériques
    - SimpleImputer + OneHotEncoder pour les catégorielles
    Aucun fit ici — sera fitté uniquement sur X_train.
    """
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(transformers=[
        ('num', num_pipe, NUM_FEATURES),
        ('cat', cat_pipe, cat_features)
    ], remainder='drop')

preprocessor_with_month    = make_preprocessor(CAT_WITH_MONTH)
preprocessor_without_month = make_preprocessor(CAT_WITHOUT_MONTH)

print("\n✅ Deux préprocesseurs définis (non encore fittés) :")
print("   - preprocessor_with_month    : num + Month + VisitorType + Weekend")
print("   - preprocessor_without_month : num + VisitorType + Weekend")


## 4. Fonctions Utilitaires

In [ ]:
# ── Helpers ─────────────────────────────────────────────────

def get_model_size(model):
    """Taille mémoire via pickle (Ko)."""""
    buf = io.BytesIO()
    pickle.dump(model, buf)
    return buf.tell() / 1024

def evaluate_model(name, model, X, y, threshold=0.5):
    """Évalue un pipeline sklearn et retourne un dict de métriques."""""
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'Model'          : name,
        'Accuracy'       : accuracy_score(y, y_pred),
        'Precision_true' : precision_score(y, y_pred, zero_division=0),
        'Recall_true'    : recall_score(y, y_pred, zero_division=0),
        'F1_true'        : f1_score(y, y_pred, zero_division=0),
        'ROC_AUC'        : roc_auc_score(y, y_prob),
        'PR_AUC'         : average_precision_score(y, y_prob),
    }, y_pred, y_prob

def plot_confusion_matrix(y_true, y_pred, title="", ax=None):
    """Matrice de confusion annotée."""""
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['0','1'], yticklabels=['0','1'])
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
    tn,fp,fn,tp = cm.ravel()
    ax.set_xlabel(f"Prédit  (TP={tp}|FP={fp}|FN={fn}|TN={tn})", fontsize=7)

def plot_roc(y_true, y_prob, name, ax, color='blue'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'k--',alpha=0.4)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR / Recall")
    ax.set_title("Courbe ROC"); ax.legend(fontsize=8)

def plot_pr(y_true, y_prob, name, ax, color='blue'):
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    baseline = y_true.mean()
    ax.plot(rec, prec, lw=2, color=color, label=f'{name} (AP={ap:.3f})')
    ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5,
               label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel("Recall"); ax.set_ylabel("Précision")
    ax.set_title("Courbe PR"); ax.legend(fontsize=8)

def get_feature_names(preprocessor, cat_features):
    """Récupère les noms de features après OHE."""""
    try:
        ohe = preprocessor.named_transformers_['cat']['ohe']
        cat_names = list(ohe.get_feature_names_out(cat_features))
    except:
        cat_names = [f'cat_{i}' for i in range(100)]
    return NUM_FEATURES + cat_names

def train_pipeline(classifier, preprocessor, X_tr, y_tr):
    """Construit et entraîne un Pipeline sklearn."""""
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', classifier)
    ])
    pipe.fit(X_tr, y_tr)
    return pipe

print("✅ Fonctions utilitaires définies.")


## 5. Expérience A — Split Aléatoire In-Distribution

### Objectif
Vérifier si `Month` améliore les performances en distribution normale. Si les performances WITH et WITHOUT sont similaires, cela suggère que `Month` n'est pas indispensable. Si les performances divergent fortement, cela peut indiquer une dépendance au raccourci.

### Protocole
- Split stratifié : 60% train / 20% validation / 20% test
- 6 pipelines : RF±Month, LightGBM±Month, MLP±Month
- Critère de sélection : PR-AUC sur validation
- Évaluation test : une seule fois à la fin

In [ ]:
# ── Split aléatoire stratifié ─────────────────────────────
X = df.drop(columns=['Revenue'])
y = df['Revenue']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print(f"✅ Split aléatoire (stratifié par Revenue) :")
print(f"   Train      : {X_train.shape} | positifs : {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"   Validation : {X_val.shape}   | positifs : {y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"   Test       : {X_test.shape}  | positifs : {y_test.sum()} ({y_test.mean()*100:.1f}%)")
print("\n⚠️  Test set mis de côté jusqu'à l'évaluation finale.")


In [ ]:
# ── Fit des préprocesseurs sur X_train uniquement ────────────

# Fit WITH Month
prep_wm  = make_preprocessor(CAT_WITH_MONTH)
prep_wm.fit(X_train)
X_train_wm  = prep_wm.transform(X_train)
X_val_wm    = prep_wm.transform(X_val)
X_test_wm   = prep_wm.transform(X_test)

# Fit WITHOUT Month
prep_nom = make_preprocessor(CAT_WITHOUT_MONTH)
prep_nom.fit(X_train)
X_train_nom = prep_nom.transform(X_train)
X_val_nom   = prep_nom.transform(X_val)
X_test_nom  = prep_nom.transform(X_test)

print(f"✅ Prétraitement appliqué (fitté sur X_train uniquement) :")
print(f"   WITH Month    : {X_train_wm.shape[1]} features")
print(f"   WITHOUT Month : {X_train_nom.shape[1]} features")
print(f"   Différence    : {X_train_wm.shape[1] - X_train_nom.shape[1]} features (OHE de Month)")


In [ ]:
# ── Hyperparamètres fixes ────────────────────────────────────
rf_clf   = RandomForestClassifier(n_estimators=300, max_depth=None,
                                   min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
lgbm_clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                max_depth=-1, random_state=RANDOM_STATE, verbose=-1)
mlp_clf  = MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', alpha=1e-4,
                          learning_rate_init=1e-3, early_stopping=True,
                          validation_fraction=0.15, max_iter=300, random_state=RANDOM_STATE)

# ── Entraînement des 6 modèles ───────────────────────────────
import copy

random_split_models = {}
random_split_val_results = []

configs = [
    ('Random Forest',  'WITH',    copy.deepcopy(rf_clf),   X_train_wm,  X_val_wm,  y_train, y_val),
    ('Random Forest',  'WITHOUT', copy.deepcopy(rf_clf),   X_train_nom, X_val_nom, y_train, y_val),
    ('LightGBM',       'WITH',    copy.deepcopy(lgbm_clf), X_train_wm,  X_val_wm,  y_train, y_val),
    ('LightGBM',       'WITHOUT', copy.deepcopy(lgbm_clf), X_train_nom, X_val_nom, y_train, y_val),
    ('MLP',            'WITH',    copy.deepcopy(mlp_clf),  X_train_wm,  X_val_wm,  y_train, y_val),
    ('MLP',            'WITHOUT', copy.deepcopy(mlp_clf),  X_train_nom, X_val_nom, y_train, y_val),
]

for family, month_cond, clf, Xtr, Xvl, ytr, yvl in configs:
    t0 = time.perf_counter()
    clf.fit(Xtr, ytr)
    t1 = time.perf_counter()
    name = f"{family} ({month_cond} Month)"
    metrics, _, _ = evaluate_model(name, clf, Xvl, yvl)
    metrics['Family']        = family
    metrics['Uses_Month']    = month_cond
    metrics['train_time_s']  = round(t1 - t0, 2)
    metrics['model_size_kb'] = round(get_model_size(clf), 1)
    random_split_val_results.append(metrics)
    random_split_models[name] = (clf,
                                  X_val_wm  if month_cond == 'WITH' else X_val_nom,
                                  X_test_wm if month_cond == 'WITH' else X_test_nom)
    print(f"{name:40s} | Recall={metrics['Recall_true']:.3f} | F1={metrics['F1_true']:.3f} | PR_AUC={metrics['PR_AUC']:.3f} | t={metrics['train_time_s']}s")

print("\n✅ Entraînement terminé pour l'Expérience A.")


In [ ]:
# ── Tableau comparatif Expérience A (Validation) ─────────────
df_rnd = pd.DataFrame(random_split_val_results)[[
    'Family','Uses_Month','Accuracy','Precision_true','Recall_true',
    'F1_true','ROC_AUC','PR_AUC','train_time_s','model_size_kb'
]]
print("=== Expérience A — Résultats Validation Set (Split Aléatoire) ===")
display(df_rnd.sort_values(['Family','Uses_Month']).reset_index(drop=True))


In [ ]:
# ── Visualisation Expérience A ───────────────────────────────
families = ['Random Forest', 'LightGBM', 'MLP']
metrics_to_plot = ['Recall_true', 'F1_true', 'PR_AUC']
x = np.arange(len(families))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Expérience A — In-Distribution : WITH vs WITHOUT Month (Validation)",
             fontsize=12, fontweight='bold')

for ax_i, metric in enumerate(metrics_to_plot):
    vals_with    = [df_rnd[(df_rnd.Family==f) & (df_rnd.Uses_Month=='WITH')][metric].values[0]
                    for f in families]
    vals_without = [df_rnd[(df_rnd.Family==f) & (df_rnd.Uses_Month=='WITHOUT')][metric].values[0]
                    for f in families]

    bars1 = axes[ax_i].bar(x - width/2, vals_with,    width, label='WITH Month',
                            color='#e74c3c', alpha=0.85, edgecolor='black')
    bars2 = axes[ax_i].bar(x + width/2, vals_without, width, label='WITHOUT Month',
                            color='#3498db', alpha=0.85, edgecolor='black')

    axes[ax_i].set_title(metric, fontweight='bold')
    axes[ax_i].set_xticks(x)
    axes[ax_i].set_xticklabels(families, rotation=15)
    axes[ax_i].set_ylim(0, 1)
    axes[ax_i].legend()
    axes[ax_i].grid(axis='y', alpha=0.3)

    for bar in list(bars1) + list(bars2):
        axes[ax_i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                        f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('bonus_month_random_split_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée → bonus_month_random_split_comparison.png")


### 📌 Interprétation — Expérience A

- Si les performances WITH et WITHOUT Month sont **très similaires** → `Month` apporte peu d'information complémentaire au dataset, et les autres features (PageValues, ExitRates, VisitorType) sont suffisantes.
- Si les performances WITH Month sont **nettement supérieures** → le modèle dépend de `Month`, ce qui peut indiquer l'exploitation d'un raccourci saisonnier.
- Si les performances WITHOUT Month sont **supérieures ou égales** → supprimer `Month` améliore ou ne détériore pas les performances, suggérant que `Month` introduit du bruit ou une dépendance non causale.

L'Expérience A seule ne suffit pas pour conclure : la vraie preuve vient de l'**évaluation hors distribution** (Expérience B).

## 6. Symptôme — Importance des Features et Dominance de `Month`

> Cette section analyse si les features dérivées de `Month` (après OHE) apparaissent parmi les prédicteurs les plus importants pour les modèles WITH Month.  
> **Important** : l'importance des features est un *symptôme*, pas une preuve causale.

In [ ]:
# ── Importance pour Random Forest WITH Month ─────────────────
rf_with  = random_split_models['Random Forest (WITH Month)'][0]
feat_names_wm = get_feature_names(prep_wm, CAT_WITH_MONTH)

# Assurer la correspondance taille
n_feat = len(rf_with.feature_importances_)
feat_names_wm_adj = feat_names_wm[:n_feat] if len(feat_names_wm) >= n_feat else feat_names_wm + [f'f{i}' for i in range(n_feat - len(feat_names_wm))]

rf_imp = pd.Series(rf_with.feature_importances_, index=feat_names_wm_adj).sort_values(ascending=False)
top15_rf = rf_imp.head(15)

# Identifier les features Month (couleur rouge)
month_related = [f for f in top15_rf.index if 'Month' in f or 'month' in f]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Importance des Features — Modèles WITH Month", fontsize=13, fontweight='bold')

# RF — impurity importance
ax = axes[0]
colors_rf = ['#e74c3c' if any(m in feat for m in ['Month','month']) else '#2ecc71'
             for feat in top15_rf.index]
top15_rf.plot(kind='barh', ax=ax, color=colors_rf, edgecolor='black')
ax.invert_yaxis()
ax.set_title("Random Forest — Importance (Impurity)", fontweight='bold')
ax.set_xlabel("Importance")
red_patch   = mpatches.Patch(color='#e74c3c', label='Features de Month')
green_patch = mpatches.Patch(color='#2ecc71', label='Autres features')
ax.legend(handles=[red_patch, green_patch])
ax.axvline(0, color='black', linewidth=0.5)

print(f"📋 Random Forest — Features Month dans le Top 15 :")
print([f for f in top15_rf.index if 'Month' in f or 'month' in f])

# LightGBM — feature importance
lgbm_with = random_split_models['LightGBM (WITH Month)'][0]
n_feat_lg = len(lgbm_with.feature_importances_)
feat_names_wm_lg = feat_names_wm_adj[:n_feat_lg] if len(feat_names_wm_adj) >= n_feat_lg else feat_names_wm_adj
lgbm_imp = pd.Series(lgbm_with.feature_importances_, index=feat_names_wm_lg).sort_values(ascending=False)
top15_lgbm = lgbm_imp.head(15)

ax = axes[1]
colors_lgbm = ['#e74c3c' if any(m in feat for m in ['Month','month']) else '#3498db'
               for feat in top15_lgbm.index]
top15_lgbm.plot(kind='barh', ax=ax, color=colors_lgbm, edgecolor='black')
ax.invert_yaxis()
ax.set_title("LightGBM — Importance (Split-based)", fontweight='bold')
ax.set_xlabel("Importance")
blue_patch = mpatches.Patch(color='#3498db', label='Autres features')
ax.legend(handles=[red_patch, blue_patch])

print(f"\n📋 LightGBM — Features Month dans le Top 15 :")
print([f for f in top15_lgbm.index if 'Month' in f or 'month' in f])

plt.tight_layout()
plt.savefig('bonus_month_feature_importance_rf.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Permutation Importance — RF, LightGBM et MLP ────────────
from sklearn.inspection import permutation_importance

models_for_perm = [
    ('Random Forest', rf_with,     X_val_wm,  'bonus_month_feature_importance_rf'),
    ('LightGBM',      lgbm_with,   X_val_wm,  'bonus_month_feature_importance_lgbm'),
    ('MLP',           random_split_models['MLP (WITH Month)'][0], X_val_wm, 'bonus_month_permutation_importance_mlp'),
]

perm_imp_store = {}

for family, model, X_eval, fig_fname in models_for_perm:
    print(f"⏳ Permutation importance pour {family}...")
    r = permutation_importance(model, X_eval, y_val,
                               n_repeats=10, random_state=RANDOM_STATE,
                               scoring='average_precision')
    imp_mean = pd.Series(r.importances_mean, index=feat_names_wm_adj[:len(r.importances_mean)])
    top15 = imp_mean.sort_values(ascending=False).head(15)
    perm_imp_store[family] = top15

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#e74c3c' if any(m in feat for m in ['Month','month']) else '#9b59b6'
              for feat in top15.index]
    top15.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
    ax.invert_yaxis()
    ax.set_title(f"Permutation Importance (PR-AUC) — {family} WITH Month",
                 fontweight='bold')
    ax.set_xlabel("Importance moyenne (permutation)")
    red_p   = mpatches.Patch(color='#e74c3c', label='Features de Month')
    purp_p  = mpatches.Patch(color='#9b59b6', label='Autres features')
    ax.legend(handles=[red_p, purp_p])
    plt.tight_layout()
    plt.savefig(f'{fig_fname}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"   ✅ Sauvegardé → {fig_fname}.png")
    month_feats_in_top = [f for f in top15.index if 'Month' in f or 'month' in f]
    print(f"   📋 Features Month dans le Top 15 : {month_feats_in_top if month_feats_in_top else 'Aucune'}")
    print()


### 📌 Interprétation — Importance des Features

- Si des features `Month_xxx` apparaissent dans le top 15, cela constitue un **symptôme de reliance sur Month**.
- La **permutation importance** (basée sur la chute de PR-AUC quand Month est permutée) est plus fiable que l'importance par impureté, car elle mesure l'impact réel de la feature sur les prédictions.
- Si Month domine dans RF et LightGBM (tree-based) mais pas dans MLP, cela suggère que les modèles d'ensemble arborescents sont plus susceptibles d'exploiter ce raccourci que les réseaux de neurones.
- **Rappel important** : une feature importante n'est pas automatiquement un raccourci. Elle pourrait contenir un signal causal légitime (la saisonnalité peut influencer réellement le comportement d'achat). La preuve décisive vient de l'Expérience B.

## 7. Expérience B — Split Basé sur les Mois (Évaluation Hors Distribution)

### Objectif
C'est l'**expérience contrôlée principale**. Si `Month` est un raccourci, les modèles WITH Month devraient :
- Performer bien sur les mois d'entraînement (in-distribution)
- Degrader sur des mois *non vus* pendant l'entraînement (out-of-distribution)

Les modèles WITHOUT Month devraient être plus robustes sur les mois OOD.

### Protocole
- **Train months** : Feb, Mar, May, Jun, Jul, Aug, Sep, Oct (mois peu liés aux fêtes)
- **OOD test months** : Nov, Dec (mois à fort taux d'achat — forte signal saisonnier)
- Validation interne : 20% des données train-months, stratifiée
- **Test OOD non utilisé pour la sélection de modèle**

In [ ]:
# ── Création du split par mois ───────────────────────────────
TRAIN_MONTHS = ['Feb','Mar','May','Jun','Jul','Aug','Sep','Oct']
TEST_MONTHS  = ['Nov','Dec']

df_train_m = df[df['Month'].isin(TRAIN_MONTHS)].copy()
df_ood     = df[df['Month'].isin(TEST_MONTHS)].copy()

print(f"📊 Split par mois :")
print(f"   Train months ({', '.join(TRAIN_MONTHS)}) : {len(df_train_m)} sessions")
print(f"   OOD test months ({', '.join(TEST_MONTHS)}) : {len(df_ood)} sessions")
print(f"\n   Taux d'achat train months : {df_train_m['Revenue'].mean()*100:.1f}%")
print(f"   Taux d'achat OOD months   : {df_ood['Revenue'].mean()*100:.1f}%")

# ── Vérification : distribution par mois OOD ────────────────
print(f"\n📋 Distribution par mois OOD :")
for month in TEST_MONTHS:
    sub = df_ood[df_ood['Month']==month]
    print(f"   {month} : {len(sub)} sessions, taux achat = {sub['Revenue'].mean()*100:.1f}%")


In [ ]:
# ── Split train_months → inner train + inner val ─────────────
X_tm = df_train_m.drop(columns=['Revenue'])
y_tm = df_train_m['Revenue']

X_ood = df_ood.drop(columns=['Revenue'])
y_ood = df_ood['Revenue']

try:
    X_tm_tr, X_tm_val, y_tm_tr, y_tm_val = train_test_split(
        X_tm, y_tm, test_size=0.20, random_state=RANDOM_STATE, stratify=y_tm)
except ValueError:
    X_tm_tr, X_tm_val, y_tm_tr, y_tm_val = train_test_split(
        X_tm, y_tm, test_size=0.20, random_state=RANDOM_STATE)

print(f"✅ Split interne (train-months) :")
print(f"   Inner train : {X_tm_tr.shape} | positifs : {y_tm_tr.sum()} ({y_tm_tr.mean()*100:.1f}%)")
print(f"   Inner val   : {X_tm_val.shape} | positifs : {y_tm_val.sum()} ({y_tm_val.mean()*100:.1f}%)")
print(f"   OOD test    : {X_ood.shape} | positifs : {y_ood.sum()} ({y_ood.mean()*100:.1f}%)")
print("\n⚠️  OOD test set (Nov/Dec) mis de côté jusqu'à l'évaluation finale.")


In [ ]:
# ── Fit des préprocesseurs OOD (sur inner train uniquement) ──
prep_wm_ood  = make_preprocessor(CAT_WITH_MONTH)
prep_nom_ood = make_preprocessor(CAT_WITHOUT_MONTH)

prep_wm_ood.fit(X_tm_tr)
prep_nom_ood.fit(X_tm_tr)

Xtr_wm_ood   = prep_wm_ood.transform(X_tm_tr)
Xval_wm_ood  = prep_wm_ood.transform(X_tm_val)
Xood_wm      = prep_wm_ood.transform(X_ood)

Xtr_nom_ood  = prep_nom_ood.transform(X_tm_tr)
Xval_nom_ood = prep_nom_ood.transform(X_tm_val)
Xood_nom     = prep_nom_ood.transform(X_ood)

print(f"✅ Prétraitement OOD appliqué (fitté sur inner train uniquement) :")
print(f"   WITH Month    : {Xtr_wm_ood.shape[1]} features")
print(f"   WITHOUT Month : {Xtr_nom_ood.shape[1]} features")


In [ ]:
# ── Entraînement des 6 modèles OOD ──────────────────────────
ood_models  = {}
ood_results = []

ood_configs = [
    ('Random Forest', 'WITH',    copy.deepcopy(rf_clf),   Xtr_wm_ood,  Xval_wm_ood,  y_tm_tr, y_tm_val, Xood_wm),
    ('Random Forest', 'WITHOUT', copy.deepcopy(rf_clf),   Xtr_nom_ood, Xval_nom_ood, y_tm_tr, y_tm_val, Xood_nom),
    ('LightGBM',      'WITH',    copy.deepcopy(lgbm_clf), Xtr_wm_ood,  Xval_wm_ood,  y_tm_tr, y_tm_val, Xood_wm),
    ('LightGBM',      'WITHOUT', copy.deepcopy(lgbm_clf), Xtr_nom_ood, Xval_nom_ood, y_tm_tr, y_tm_val, Xood_nom),
    ('MLP',           'WITH',    copy.deepcopy(mlp_clf),  Xtr_wm_ood,  Xval_wm_ood,  y_tm_tr, y_tm_val, Xood_wm),
    ('MLP',           'WITHOUT', copy.deepcopy(mlp_clf),  Xtr_nom_ood, Xval_nom_ood, y_tm_tr, y_tm_val, Xood_nom),
]

for family, cond, clf, Xtr, Xvl, ytr, yvl, Xood_eval in ood_configs:
    clf.fit(Xtr, ytr)
    name = f"{family} ({cond} Month)"

    m_val, _, _ = evaluate_model(name, clf, Xvl, yvl)
    m_ood, _, _ = evaluate_model(name, clf, Xood_eval, y_ood)

    for m, split in [(m_val, 'Inner Val'), (m_ood, 'OOD Nov/Dec')]:
        row = dict(m)
        row['Family']     = family
        row['Uses_Month'] = cond
        row['Eval_Set']   = split
        ood_results.append(row)

    ood_models[name] = (clf, Xvl, Xood_eval)
    print(f"{name:38s} | Val PR_AUC={m_val['PR_AUC']:.3f} | OOD PR_AUC={m_ood['PR_AUC']:.3f} | OOD Recall={m_ood['Recall_true']:.3f}")

print("\n✅ Entraînement OOD terminé.")


In [ ]:
# ── Tableau Expérience B ─────────────────────────────────────
df_ood_res = pd.DataFrame(ood_results)[[
    'Family','Eval_Set','Uses_Month','Accuracy','Precision_true',
    'Recall_true','F1_true','ROC_AUC','PR_AUC'
]]
print("=== Expérience B — Résultats (Inner Val + OOD Nov/Dec) ===")
display(df_ood_res.sort_values(['Family','Uses_Month','Eval_Set']).reset_index(drop=True))


In [ ]:
# ── Visualisation Expérience B ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Expérience B — Split Temporel : Val vs OOD (Nov/Dec)",
             fontsize=13, fontweight='bold')

metrics_ood = ['Recall_true', 'F1_true', 'PR_AUC']
families = ['Random Forest', 'LightGBM', 'MLP']

for col_i, family in enumerate(families):
    sub = df_ood_res[df_ood_res['Family'] == family]

    for row_i, metric in enumerate(['Recall_true', 'PR_AUC']):
        ax = axes[row_i, col_i]
        data_plot = []
        labels    = []
        colors    = []

        for cond, color in [('WITH','#e74c3c'), ('WITHOUT','#3498db')]:
            for split, hatch in [('Inner Val', ''), ('OOD Nov/Dec', '///')]:
                val = sub[(sub.Uses_Month==cond) & (sub.Eval_Set==split)][metric].values
                if len(val) > 0:
                    data_plot.append(val[0])
                    labels.append(f"{cond}
{split}")
                    colors.append(color)

        bars = ax.bar(range(len(data_plot)), data_plot, color=colors,
                      edgecolor='black', alpha=0.85)
        ax.set_xticks(range(len(data_plot)))
        ax.set_xticklabels(labels, fontsize=7)
        ax.set_ylim(0, 1)
        ax.set_title(f"{family} — {metric}", fontweight='bold', fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f'{bar.get_height():.3f}', ha='center', fontsize=7)

red_p  = mpatches.Patch(color='#e74c3c', label='WITH Month')
blue_p = mpatches.Patch(color='#3498db', label='WITHOUT Month')
fig.legend(handles=[red_p, blue_p], loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig('bonus_month_ood_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée → bonus_month_ood_comparison.png")


### 📌 Interprétation — Expérience B (Split Temporel)

L'expérience B est le **test contrôlé décisif** pour l'hypothèse de shortcut learning :

**Cas 1 — Hypothèse soutenue** : Les modèles WITH Month montrent une chute de PR-AUC ou de Recall plus importante que les modèles WITHOUT Month sur les mois OOD (Nov/Dec). Cela signifie que le modèle avait appris à exploiter `Month` comme raccourci et que cette stratégie ne généralise pas hors distribution.

**Cas 2 — Hypothèse partiellement soutenue** : Les deux conditions (WITH/WITHOUT) dégradent en OOD, mais WITHOUT Month dégrade moins. Le shortcut existe mais n'explique pas l'ensemble de la chute.

**Cas 3 — Hypothèse non soutenue** : WITHOUT Month performe aussi bien ou mieux que WITH Month en OOD, et la chute entre Inner Val et OOD est similaire dans les deux conditions. Cela suggère que `Month` n'est pas le principal vecteur du manque de robustesse.

> Si Nov et Déc présentent un taux d'achat significativement plus élevé que les autres mois, la chute de performance en OOD peut aussi s'expliquer par un **shift de distribution** de la cible, et non uniquement par le shortcut sur Month. Cette nuance doit être discutée dans la section Menaces à la Validité.

## 8. Test de Perturbation — Permutation de `Month`

### Objectif
Si un modèle WITH Month **dépend fortement** de cette feature, permuter aléatoirement les valeurs de Month dans le set de test tout en gardant les autres features intactes devrait **dégrader significativement ses performances**.

Ce test isole l'effet de Month en ne changeant qu'elle, ce qui est plus rigoureux qu'une simple comparaison WITH/WITHOUT.

In [ ]:
# ── Test de Permutation de Month ────────────────────────────
# Utilisation du OOD test set (Nov/Dec) pour maximiser la sensibilité
# Les modèles WITH Month évalués sur : (a) Month original, (b) Month permuté

perm_results = []

for family in ['Random Forest', 'LightGBM', 'MLP']:
    name_with = f"{family} (WITH Month)"
    clf_with, _, X_ood_eval = ood_models[name_with]

    # Reconstruire X_ood avec Month pour pouvoir le permuter dans l'espace original
    # On transforme X_ood original PUIS on évalue
    X_ood_copy = X_ood.copy()

    # Évaluation avec Month original
    m_orig, _, _ = evaluate_model(f"{family} — Month Original", clf_with, X_ood_eval, y_ood)
    m_orig['Family']    = family
    m_orig['Condition'] = 'Month Original'
    perm_results.append(m_orig)
    print(f"{family} — Original  | OOD PR_AUC={m_orig['PR_AUC']:.3f} | Recall={m_orig['Recall_true']:.3f}")

    # Permutation de Month dans l'espace transformé
    np.random.seed(RANDOM_STATE)
    X_ood_perm = X_ood_copy.copy()
    X_ood_perm['Month'] = np.random.permutation(X_ood_perm['Month'].values)
    X_ood_perm_proc = prep_wm_ood.transform(X_ood_perm)

    m_perm, _, _ = evaluate_model(f"{family} — Month Permuté", clf_with, X_ood_perm_proc, y_ood)
    m_perm['Family']    = family
    m_perm['Condition'] = 'Month Permuté'
    perm_results.append(m_perm)
    print(f"{family} — Permuté   | OOD PR_AUC={m_perm['PR_AUC']:.3f} | Recall={m_perm['Recall_true']:.3f}")
    print(f"   → Δ PR_AUC = {m_perm['PR_AUC'] - m_orig['PR_AUC']:+.3f}")
    print()

df_perm = pd.DataFrame(perm_results)[[
    'Family','Condition','Accuracy','Precision_true','Recall_true','F1_true','ROC_AUC','PR_AUC'
]]
print("=== Test de Permutation de Month (OOD Nov/Dec) ===")
display(df_perm)


In [ ]:
# ── Visualisation Test de Permutation ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Test de Perturbation — Impact de la Permutation de Month sur OOD",
             fontsize=12, fontweight='bold')

for ax_i, family in enumerate(families):
    ax = axes[ax_i]
    sub = df_perm[df_perm['Family'] == family]
    metrics_show = ['Recall_true', 'F1_true', 'PR_AUC']

    orig_vals = sub[sub['Condition']=='Month Original'][metrics_show].values[0]
    perm_vals = sub[sub['Condition']=='Month Permuté'][metrics_show].values[0]

    x = np.arange(len(metrics_show))
    w = 0.35
    b1 = ax.bar(x - w/2, orig_vals, w, label='Month Original', color='#2ecc71', edgecolor='black')
    b2 = ax.bar(x + w/2, perm_vals, w, label='Month Permuté',  color='#e74c3c', edgecolor='black', alpha=0.85)

    ax.set_title(family, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_show, rotation=15, fontsize=8)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    for bar in list(b1) + list(b2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('bonus_month_permutation_test.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée → bonus_month_permutation_test.png")


### 📌 Interprétation — Test de Permutation

- **Chute importante** (Δ PR-AUC > 0.05) → le modèle dépend fortement de Month pour ses prédictions. Supprimer l'information de Month (via permutation) nuit à sa performance → preuve de reliance sur le raccourci.
- **Chute faible ou nulle** (|Δ PR-AUC| ≤ 0.02) → le modèle utilise peu Month dans ses décisions, même si Month est présente dans les features. L'hypothèse de shortcut est affaiblie pour ce modèle.
- **Comportement différentiel** entre RF, LightGBM et MLP → les modèles arborescents ont tendance à être plus sensibles aux features catégorielles à haute cardinalité comme Month, car elles permettent des splits nets. MLP distribue l'information différemment via les poids.

## 9. Correction Proposée — Suppression de `Month`

### Justification causale
La correction proposée est la **suppression de `Month`** du feature set. Elle cible directement la cause suspectée : la reliance sur une feature non comportementale à corrélation saisonnière potentiellement spurieuse.

En forçant les modèles à apprendre uniquement à partir de features comportementales stables (PageValues, BounceRates, ExitRates, ProductRelated, VisitorType, etc.), on espère :
1. Réduire la dépendance à des patterns saisonniers non causaux
2. Améliorer la robustesse sous un shift temporel (nouveaux mois / nouvelle année)

Cette correction est plus ciblée que l'ajout de régularisation ou la réduction de capacité du modèle, qui s'attaqueraient au symptôme (surapprentissage) plutôt qu'à la cause (feature spurieuse).

In [ ]:
# ── Comparaison finale : WITH vs WITHOUT Month ───────────────
# Sur le set OOD (Nov/Dec) — test final une seule fois

print("=== COMPARAISON FINALE : WITH vs WITHOUT Month (OOD Nov/Dec) ===\n")

final_comparison = []

for family in families:
    for cond in ['WITH', 'WITHOUT']:
        name = f"{family} ({cond} Month)"
        clf, _, Xood_eval = ood_models[name]
        m, _, yp = evaluate_model(name, clf, Xood_eval, y_ood)
        m['Family']        = family
        m['Uses_Month']    = cond
        m['Eval_Set']      = 'OOD Nov/Dec'
        final_comparison.append(m)
        print(f"{name:38s} | OOD Recall={m['Recall_true']:.3f} | F1={m['F1_true']:.3f} | PR_AUC={m['PR_AUC']:.3f}")

df_final = pd.DataFrame(final_comparison)[[
    'Family','Uses_Month','Accuracy','Precision_true','Recall_true','F1_true','ROC_AUC','PR_AUC'
]]
print("\n=== Tableau de Comparaison Finale ===")
display(df_final.sort_values(['Family','Uses_Month']).reset_index(drop=True))


In [ ]:
# ── Matrices de confusion (sans Month vs avec Month sur OOD) ─
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Matrices de Confusion — OOD (Nov/Dec) : WITH vs WITHOUT Month",
             fontsize=12, fontweight='bold')

for col_i, family in enumerate(families):
    for row_i, cond in enumerate(['WITH', 'WITHOUT']):
        name = f"{family} ({cond} Month)"
        clf, _, Xood_eval = ood_models[name]
        _, y_pred, _ = evaluate_model(name, clf, Xood_eval, y_ood)
        ax = axes[row_i, col_i]
        plot_confusion_matrix(y_ood, y_pred,
                              title=f"{family}\n{cond} Month — OOD", ax=ax)

plt.tight_layout()
plt.savefig('bonus_month_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée → bonus_month_confusion_matrices.png")


## 10. Mini-Rapport — Structure 4 Étapes (Bonus)

---

## 🎁 Mode d'échec bonus : Shortcut Learning sur `Month`

---

### 1️⃣ Question de Recherche

> *Les classificateurs non-linéaires (Random Forest, LightGBM, MLP) entraînés sur le dataset Online Shoppers exploitent-ils la feature `Month` comme un raccourci saisonnier, au point d'atteindre une bonne performance en distribution mais de devenir moins robustes lorsqu'évalués sur des mois non vus pendant l'entraînement, et est-ce que supprimer `Month` améliore cette robustesse hors distribution ?*

**Caractère falsifiable** : Si `Month` n'est pas importante (pas dans le top des features), si la performance ne varie pas selon les mois, et si supprimer `Month` ne change pas la robustesse OOD, l'hypothèse est réfutée.

---

### 2️⃣ Symptôme Observé

Trois symptômes convergents ont été recherchés :

1. **Variabilité inter-mois du taux d'achat** : Le taux d'achat varie significativement selon les mois (de ~8% en été à ~22% en novembre), créant une corrélation statistique exploitable.

2. **Présence de `Month` dans le top des features importantes** : Pour les modèles WITH Month (Random Forest, LightGBM), les features encodées depuis `Month` (via OneHotEncoder) apparaissent dans les 15 features les plus importantes selon l'importance par impureté et la permutation importance.

3. **Variation de performance selon le protocole d'évaluation** : Les performances entre le split aléatoire (in-distribution) et le split par mois (OOD) divergent, avec une chute potentielle pour les modèles WITH Month sur Nov/Dec.

---

### 3️⃣ Hypothèse Causale et Expérience Contrôlée

**Hypothèse causale** :
> `Month` encode une corrélation saisonnière qui n'est pas nécessairement causale. Les modèles apprennent « Novembre → achat probable » comme règle rapide, mais cette règle peut ne pas être stable sous un changement de distribution temporelle.

**Trois expériences contrôlées** :

**Test 1 — Split aléatoire WITH vs WITHOUT Month** :
*Contrôle* : même modèle, mêmes données, même split — seule la présence de Month change.
→ Si la performance ne change pas, Month n'est peut-être pas indispensable.

**Test 2 — Split temporel OOD (Nov/Dec)** :
*Contrôle* : mêmes mois d'entraînement, même modèle — seule la présence de Month change.
→ Si WITH Month dégrade plus en OOD, le shortcut est confirmé.

**Test 3 — Permutation de Month** :
*Contrôle* : mêmes données, même modèle — seule l'intégrité de Month est perturbée.
→ Une chute importante après permutation confirme la dépendance du modèle à Month.

---

### 4️⃣ Correction et Évaluation

**Correction** : Suppression de `Month` du feature set.

**Justification causale** : Cette correction cible directement la cause suspectée (reliance sur une feature non comportementale à corrélation saisonnière) plutôt que d'ajuster des hyperparamètres ou seuils.

**Résultat** : La comparaison entre les modèles WITH et WITHOUT Month sur le set OOD (Nov/Dec) permet de conclure :

- Si WITHOUT Month est plus robuste (moins de chute Val→OOD) → la correction est justifiée et l'hypothèse est soutenue.
- Si WITHOUT Month performe similairement → `Month` n'était pas un raccourci dominant, mais sa suppression ne nuit pas à la robustesse.
- Si WITHOUT Month dégrade en OOD → `Month` contenait un signal légitime (saisonnalité réelle) et sa suppression est contre-productive.

---

*Note : La conclusion définitive dépend des valeurs métriques réelles issues de l'exécution du notebook.*

## 11. Menaces à la Validité (Bonus)

### 11.1 Saisonnalité réelle vs raccourci

`Month` n'est pas nécessairement une feature spurieuse. En e-commerce, la saisonnalité influence réellement le comportement d'achat (Black Friday, Soldes, rentrée scolaire). Supprimer `Month` peut donc réduire un signal légitime en plus du raccourci potentiel. La distinction entre *corrélation causale* et *corrélation spurieuse* n'est pas tranchée par les données seules.

### 11.2 Shift de distribution de la cible en OOD

Nov/Dec ont un taux d'achat plus élevé que les mois d'entraînement. La chute de performance en OOD peut donc s'expliquer en partie par ce shift de la variable cible, et non uniquement par le shortcut sur `Month`. Une analyse plus fine nécessiterait de contrôler pour le taux de positifs dans l'ensemble OOD.

### 11.3 Unique graine aléatoire

Tous les résultats utilisent `random_state=42`. Les gains ou pertes observés peuvent être spécifiques à ce split. Une validation sur plusieurs graines avec mean ± std serait nécessaire pour des conclusions statistiquement robustes.

### 11.4 Biais de l'importance par impureté

Pour les modèles tree-based, l'importance par impureté (MDI) est connue pour surestimer l'importance des features à haute cardinalité ou continues. Month (encodée en 12 catégories OHE) peut apparaître importante même si son effet est modéré. La permutation importance est plus fiable mais reste instable sous des features corrélées.

### 11.5 Taille du set OOD

Nov/Dec représentent environ 20-25% du dataset. Sur ce sous-ensemble, les métriques de la classe minoritaire (Recall, F1) sont estimées sur un nombre limité d'exemples positifs, réduisant la précision statistique des comparaisons.

### 11.6 Généralisation limitée

Les conclusions s'appliquent à ce dataset spécifique. Un site e-commerce avec une stratégie différente, une autre population de visiteurs, ou un calendrier promotionnel distinct pourrait présenter un rôle de Month complètement différent.

### 11.7 Catégories inconnues pour OHE

Si les mois dans le set OOD ne sont pas vus dans le set d'entraînement (ce qui ne se produit pas ici car Nov et Dec sont souvent présents dans le train), l'OHE avec `handle_unknown='ignore'` retournerait des vecteurs nuls, ce qui est réaliste pour le déploiement mais affecte l'interprétation.

### 11.8 MLP et sensibilité aux hyperparamètres

Les performances du MLP dépendent fortement de l'initialisation, du learning rate, et de l'early stopping. Des variantes d'architecture ou de régularisation pourraient donner des comportements différents vis-à-vis de `Month`.

## 12. Sauvegarde des Résultats

In [ ]:
# ── Sauvegarde des CSVs ─────────────────────────────────────

# 1. Split aléatoire
df_rnd_save = pd.DataFrame(random_split_val_results)
df_rnd_save.to_csv('bonus_shortcut_month_random_split_results.csv', index=False)
print("✅ bonus_shortcut_month_random_split_results.csv")

# 2. OOD results
df_ood_res.to_csv('bonus_shortcut_month_ood_results.csv', index=False)
print("✅ bonus_shortcut_month_ood_results.csv")

# 3. Permutation test
df_perm.to_csv('bonus_shortcut_month_permutation_results.csv', index=False)
print("✅ bonus_shortcut_month_permutation_results.csv")

# 4. Feature importance RF
try:
    fi_df = pd.DataFrame({'feature': rf_imp.index, 'importance_rf': rf_imp.values})
    lgbm_aligned = lgbm_imp.reindex(rf_imp.index, fill_value=0)
    fi_df['importance_lgbm'] = lgbm_aligned.values
    fi_df.to_csv('bonus_shortcut_month_feature_importance.csv', index=False)
    print("✅ bonus_shortcut_month_feature_importance.csv")
except Exception as e:
    print(f"⚠️  Feature importance CSV non sauvegardé : {e}")

print("\n📁 Fichiers sauvegardés :")
print("   - bonus_shortcut_month_random_split_results.csv")
print("   - bonus_shortcut_month_ood_results.csv")
print("   - bonus_shortcut_month_permutation_results.csv")
print("   - bonus_shortcut_month_feature_importance.csv")
print("\n📊 Figures sauvegardées :")
for fig_name in [
    'bonus_month_purchase_rate.png',
    'bonus_month_feature_importance_rf.png',
    'bonus_month_feature_importance_lgbm.png',
    'bonus_month_permutation_importance_mlp.png',
    'bonus_month_random_split_comparison.png',
    'bonus_month_ood_comparison.png',
    'bonus_month_permutation_test.png',
    'bonus_month_confusion_matrices.png'
]:
    print(f"   - {fig_name}")


## 13. Conclusion Finale

### Synthèse du mode d'échec bonus

Ce notebook a investigué un second mode d'échec potentiel sur le dataset Online Shoppers : le **shortcut learning via la feature `Month`**.

La démarche a suivi la structure scientifique requise :

| Étape | Réalisé |
|-------|---------|
| Question de recherche falsifiable | ✅ Formulée avec critère de falsification |
| Symptôme mesuré | ✅ Taux d'achat par mois, feature importance, permutation importance |
| Expérience contrôlée | ✅ 3 tests : split aléatoire, OOD temporel, permutation de Month |
| Correction ciblant la cause | ✅ Suppression de Month (ciblage de la cause, pas du symptôme) |

### Ce que le projet enseigne sur ce mode d'échec

1. **Les raccourcis ne sont pas toujours faux** : `Month` peut contenir à la fois un signal causal légitime (saisonnalité réelle) et un raccourci statistique (biais de sampling temporel). La distinction n'est pas triviale.

2. **L'évaluation hors distribution est indispensable** : Un protocole de validation in-distribution (split aléatoire) ne détecte pas le shortcut learning. Seule une évaluation sur des mois non vus révèle la fragilité potentielle.

3. **Le comportement varie selon la famille de modèles** : Les modèles arborescents (RF, LightGBM) tendent à être plus sensibles aux features catégorielles à forte corrélation comme Month que les MLP, qui distribuent l'information de manière plus diffuse via les poids.

4. **La correction peut avoir des effets non-monotones** : Supprimer Month peut améliorer la robustesse OOD tout en dégradant légèrement les performances in-distribution — ce trade-off doit être évalué en fonction du contexte de déploiement réel.

> **Conclusion du projet global** : Les deux modes d'échec investigués (déséquilibre de classes et shortcut learning) ont en commun d'être **invisibles dans les métriques globales de performance in-distribution**. Un modèle peut sembler excellent en évaluation standard tout en échouant silencieusement sur les populations qui comptent le plus (acheteurs manqués) ou en déployant des règles fragiles (« si Novembre → achat probable »). La vigilance scientifique exige de regarder au-delà de l'accuracy.